# Questão 4 — Destilação de Conhecimento (Chain-of-Thought Distillation)
### Teacher: Qwen/Qwen2.5-3B → Student: Qwen/Qwen2.5-1.5B-Instruct
### Dataset: benchmark_diarios.json (100 perguntas sobre DOMPI-2025)

**Fluxo:**
1. **Parte 1 — Professor**: o Qwen2.5-3B processa as 100 perguntas do benchmark e gera `reasoning + answer` → salva `distillation_dataset.jsonl`
2. **Parte 2 — Avaliação do Aluno ANTES**: métricas quantitativas (PPL, EC, Acurácia) e respostas qualitativas do Qwen2.5-1.5B sem fine-tuning
3. **Parte 3 — Treino do Aluno**: fine-tuning com LoRA + SFTTrainer sobre o dataset gerado pelo professor
4. **Parte 4 — Avaliação do Aluno DEPOIS**: mesmas métricas e respostas qualitativas pós-destilação
5. **Parte 5 — Comparação Final**: teacher vs aluno antes vs aluno depois — análise de transferência de conhecimento

> Todos os artefatos gerados (datasets intermediários, métricas, resultados) são lidos e salvos dentro de `RESULTS_DIR`, definido na célula de configuração abaixo.

---
# PARTE 1 — Professor (Qwen2.5-3B)
O professor processa o benchmark e gera raciocínio passo a passo para cada pergunta.

## 1.1 — Instalação

In [ ]:
!pip install -q transformers datasets torch tqdm peft trl bitsandbytes accelerate

## 1.2 — Configuração, Benchmark e Carregamento do Professor

In [ ]:
import json
import math
import os
import re

import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Configuração geral ──────────────────────────────────────────────────────
RESULTS_DIR  = "results_q4_destilação"
TEACHER_NAME = "Qwen/Qwen2.5-3B"
STUDENT_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  

os.makedirs(RESULTS_DIR, exist_ok=True)


def result_path(filename):
    """Monta o caminho de um arquivo dentro da pasta de resultados (RESULTS_DIR)."""
    return os.path.join(RESULTS_DIR, filename)


# Carrega benchmark
benchmark = load_dataset("json", data_files=result_path("benchmark_diarios.json"), split="train")
print(f"\nBenchmark carregado: {len(benchmark)} perguntas")
print("\nExemplo:")
print(f"  instruction: {benchmark[0]['instruction']}")
print(f"  input:       {benchmark[0]['input'][:80]}...")
print(f"  output:      {benchmark[0]['output']}")

# Carrega professor
print(f"\nCarregando professor ({TEACHER_NAME})...")
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME)
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
teacher_model.eval()
n_params = sum(p.numel() for p in teacher_model.parameters())
print(f"Professor carregado. Parâmetros: {n_params/1e9:.2f}B")

## 1.3 — Professor Gera reasoning + answer (Dataset de Destilação)

In [ ]:
def build_prompt(example):
    """
    Prompt no mesmo padrão do cot_distillation.ipynb do professor:
    instrui o teacher a retornar JSON com reasoning e answer.
    """
    return f"""You are a teacher model generating high-quality training data.

Return ONLY valid JSON, no extra text:
{{
  "reasoning": "step-by-step reasoning here",
  "answer": "final answer here"
}}

Rules:
- Output MUST be valid JSON only
- Reasoning must be clear and structured
- Answer must be concise and direct

Task: {example.get("instruction", "")}

Document:
{example.get("input", "")}"""


def safe_parse(text):
    """Extrai JSON da resposta, mesmo com texto extra ao redor."""
    try:
        parsed = json.loads(text)
    except Exception:
        start = text.find("{")
        end   = text.rfind("}")
        if start != -1 and end != -1:
            try:
                parsed = json.loads(text[start:end + 1])
            except Exception:
                return {"reasoning": "", "answer": text.strip()}
        else:
            return {"reasoning": "", "answer": text.strip()}

    # Se vier como lista, pega o primeiro item (se for dict) ou trata como falha
    if isinstance(parsed, list):
        if len(parsed) > 0 and isinstance(parsed[0], dict):
            return parsed[0]
        return {"reasoning": "", "answer": text.strip()}

    if not isinstance(parsed, dict):
        return {"reasoning": "", "answer": text.strip()}

    return parsed


results = []
falhas  = 0

print(f"Gerando dataset de destilação com o professor ({len(benchmark)} exemplos)...")

for i, example in enumerate(tqdm(benchmark, desc="Teacher gerando")):
    messages = [{"role": "user", "content": build_prompt(example)}]
    text = teacher_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = teacher_tokenizer([text], return_tensors="pt").to(teacher_model.device)

    with torch.no_grad():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=teacher_tokenizer.eos_token_id,
        )

    gen_text = teacher_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )

    parsed = safe_parse(gen_text)

    results.append({
        "instruction": example.get("instruction"),
        "input":       example.get("input"),
        "output":      example.get("output"),   # resposta de referência do benchmark
        "reasoning":   parsed.get("reasoning", ""),
        "answer":      parsed.get("answer", gen_text.strip()),
    })

    if not parsed.get("reasoning"):
        falhas += 1

# Salva dataset de destilação
OUTPUT_PATH = result_path("distillation_dataset.jsonl")
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"\nDataset de destilação salvo: {OUTPUT_PATH}")
print(f"Total: {len(results)} exemplos | Falhas de parse JSON: {falhas}")
print("\nExemplo gerado pelo professor:")
ex = results[0]
print(f"  Instrução : {ex['instruction']}")
print(f"  Reasoning : {ex['reasoning'][:200]}")
print(f"  Answer    : {ex['answer'][:100]}")

# Libera VRAM do professor antes de carregar o aluno
print("\nDescarregando professor da VRAM...")
del teacher_model
torch.cuda.empty_cache()
print("VRAM liberada.")

---
# PARTE 2 — Avaliação do Aluno ANTES da Destilação
Carrega o Qwen2.5-1.5B sem fine-tuning e avalia com as métricas.
Perplexidade (PPL)
Entropia Cruzada (EC)
Acurácia de Previsão de Tokens

## 2.1 — Carrega o Aluno (baseline)

In [ ]:
print(f"Carregando aluno ({STUDENT_NAME})...")
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME)
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

# Carrega sem device_map e move explicitamente para cuda:0
# Isso evita o RuntimeError de gradient em meta device durante o treino LoRA
student_base = AutoModelForCausalLM.from_pretrained(
    STUDENT_NAME,
    torch_dtype=torch.float16,
)
student_base = student_base.to("cuda:0")
student_base.eval()

# Confirma que nenhuma camada está em meta
devices = set(str(p.device) for p in student_base.parameters())
print(f"Devices do aluno: {devices}")  # deve mostrar só {'cuda:0'}

n_params = sum(p.numel() for p in student_base.parameters())
print(f"Aluno carregado. Parâmetros: {n_params/1e6:.1f}M")

## 2.2 — Funções de Avaliação Quantitativa (PPL, EC, Acurácia)

In [ ]:
def format_for_eval(example):
    """Formata o exemplo no mesmo padrão do treino, para avaliação consistente."""
    return (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Document:\n{example['input']}\n\n"
        f"### Response:\n"
        f"Reasoning:\n{example['reasoning']}\n\n"
        f"Answer:\n{example['answer']}"
    )


def calcular_metricas(modelo, tokenizador, exemplos, batch_size=4, max_length=512):
    """
    Calcula as 3 métricas:
    - Perplexidade (PPL)
    - Entropia Cruzada (EC)
    - Acurácia de Previsão de Tokens
    """
    modelo.eval()
    total_loss, total_tokens, total_certos = 0.0, 0, 0

    textos = [format_for_eval(ex) for ex in exemplos]

    for i in range(0, len(textos), batch_size):
        batch = textos[i:i + batch_size]
        enc = tokenizador(
            batch, return_tensors="pt",
            max_length=max_length, truncation=True, padding="max_length"
        )
        input_ids      = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        with torch.no_grad():
            out = modelo(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        n_tok = attention_mask.sum().item()
        total_loss   += out.loss.item() * n_tok
        total_tokens += n_tok

        preds = out.logits.argmax(dim=-1)
        mask_shift = attention_mask[:, 1:].bool()
        total_certos += (preds[:, :-1] == input_ids[:, 1:])[mask_shift].sum().item()

    ec  = total_loss / total_tokens
    ppl = math.exp(min(ec, 20))   # cap para evitar overflow em modelos não treinados
    acc = total_certos / total_tokens * 100

    return {
        "entropia_cruzada": round(ec, 4),
        "perplexidade":     round(ppl, 2),
        "acuracia_tokens":  round(acc, 2)
    }


def gerar_resposta(modelo, tokenizador, instruction, input_text, max_new_tokens=200):
    user_content = (
        f"Responda de forma concisa à pergunta abaixo com base no documento.\n\n"
        f"Pergunta: {instruction}\n\n"
        f"Documento:\n{input_text}"
    )
    messages = [{"role": "user", "content": user_content}]
    text = tokenizador.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizador([text], return_tensors="pt", truncation=True, max_length=600).to(DEVICE)

    with torch.no_grad():
        out = modelo.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,           # evita loop determinístico
            temperature=0.3,          # baixo = mais factual, menos criativo
            top_p=0.9,                # nucleus sampling
            repetition_penalty=1.3,   # penaliza repetição de tokens
            pad_token_id=tokenizador.eos_token_id,
            eos_token_id=tokenizador.eos_token_id,
        )
    return tokenizador.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

## 2.3 — Métricas Quantitativas do Aluno ANTES

In [ ]:
# Carrega o dataset gerado pelo professor para avaliação
with open(result_path("distillation_dataset.jsonl"), encoding="utf-8") as f:
    dataset_distilacao = [json.loads(l) for l in f if l.strip()]
print(f"Dataset de destilação carregado: {len(dataset_distilacao)} exemplos")

In [ ]:
print("Calculando métricas do aluno ANTES da destilação...")
metricas_aluno_antes = calcular_metricas(student_base, student_tokenizer, dataset_distilacao)

print("\n=== ALUNO — ANTES DA DESTILAÇÃO ===")
print(f"  Entropia Cruzada : {metricas_aluno_antes['entropia_cruzada']}")
print(f"  Perplexidade     : {metricas_aluno_antes['perplexidade']}")
print(f"  Acurácia Tokens  : {metricas_aluno_antes['acuracia_tokens']}%")

with open(result_path("metricas_aluno_antes.json"), "w", encoding="utf-8") as f:
    json.dump(metricas_aluno_antes, f, ensure_ascii=False, indent=2)
print(f"\nMétricas salvas em {result_path('metricas_aluno_antes.json')}")

## 2.4 — Respostas Qualitativas do Aluno ANTES

In [ ]:
print("--- Respostas qualitativas do aluno ANTES ---")
respostas_aluno_antes = []

for i, ex in enumerate(dataset_distilacao, start=1):
    resp = gerar_resposta(student_base, student_tokenizer, ex["instruction"], ex["input"])
    respostas_aluno_antes.append({
        "id":                   i,
        "instruction":          ex["instruction"],
        "input":                ex["input"][:200],
        "output_esperado":      ex["output"],
        "reasoning_professor":  ex["reasoning"],
        "answer_professor":     ex["answer"],
        "resposta_aluno_antes": resp,
    })

# Mostra 5 exemplos
for r in respostas_aluno_antes[:5]:
    print(f"Pergunta  : {r['instruction']}")
    print(f"Esperado  : {r['output_esperado']}")
    print(f"Professor : {r['answer_professor'][:120]}")
    print(f"Aluno     : {r['resposta_aluno_antes'][:120]}")
    print()

with open(result_path("respostas_aluno_antes.json"), "w", encoding="utf-8") as f:
    json.dump(respostas_aluno_antes, f, ensure_ascii=False, indent=2)
print(f"Respostas do aluno ANTES salvas em {result_path('respostas_aluno_antes.json')} ({len(respostas_aluno_antes)} exemplos)")

---
# PARTE 3 — Fine-Tuning do Aluno com LoRA + SFTTrainer


## 3.1 — Formata Dataset e Configura LoRA

In [ ]:
from datasets import load_dataset as hf_load
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments

# Carrega e formata o dataset — mesmo padrão do student_training.ipynb do professor
ds_treino = hf_load("json", data_files=result_path("distillation_dataset.jsonl"), split="train")


def format_func(example):
    """Mesmo formato do student_training.ipynb do professor."""
    return (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Document:\n{example['input']}\n\n"
        f"### Response:\n"
        f"Reasoning:\n{example['reasoning']}\n\n"
        f"Answer:\n{example['answer']}"
    )


print(f"Dataset de treino: {len(ds_treino)} exemplos")
print("\nExemplo formatado:")
print(format_func(ds_treino[0])[:400])
print("...")

# Configura LoRA — mesmos hiperparâmetros do professor
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]   # padrão Qwen (igual ao professor)
)

student_model = get_peft_model(student_base, lora_config)
student_model.print_trainable_parameters()

## 3.2 — Treinamento com SFTTrainer

In [ ]:
from trl import SFTConfig, SFTTrainer

LORA_OUTPUT_DIR = result_path("aluno-cot-lora")

training_args = SFTConfig(
    output_dir=LORA_OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=5,
    logging_first_step=True,
    save_steps=200,
    fp16=(DEVICE == "cuda"),
    bf16=False,
    dataloader_num_workers=0,
    report_to="none",
    max_length=512,
)

trainer = SFTTrainer(
    model=student_model,
    train_dataset=ds_treino,
    args=training_args,
    formatting_func=format_func,
)

print("CUDA disponível:", torch.cuda.is_available())
print("Device do modelo:", next(student_model.parameters()).device)
print("\nIniciando destilação (LoRA + SFTTrainer)...\n")

resultado = trainer.train()

print("\nTreinamento concluído!")
print(f"  Loss final : {resultado.training_loss:.4f}")
print(f"  Tempo total: {resultado.metrics['train_runtime']:.0f}s")

# Salva os adaptadores LoRA
student_model.save_pretrained(LORA_OUTPUT_DIR)
student_tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"\nAdaptadores LoRA salvos em {LORA_OUTPUT_DIR}")

---
# PARTE 4 — Avaliação do Aluno DEPOIS da Destilação

## 4.1 — Métricas Quantitativas do Aluno DEPOIS

In [ ]:
print("Calculando métricas do aluno APÓS destilação...")
metricas_aluno_depois = calcular_metricas(student_model, student_tokenizer, dataset_distilacao)

print("\n=== ALUNO — DEPOIS DA DESTILAÇÃO ===")
print(f"  Entropia Cruzada : {metricas_aluno_depois['entropia_cruzada']}")
print(f"  Perplexidade     : {metricas_aluno_depois['perplexidade']}")
print(f"  Acurácia Tokens  : {metricas_aluno_depois['acuracia_tokens']}%")

with open(result_path("metricas_aluno_depois.json"), "w", encoding="utf-8") as f:
    json.dump(metricas_aluno_depois, f, ensure_ascii=False, indent=2)
print(f"\nMétricas salvas em {result_path('metricas_aluno_depois.json')}")

## 4.2 — Respostas Qualitativas do Aluno DEPOIS

In [ ]:
print("--- Respostas qualitativas do aluno DEPOIS ---")
respostas_aluno_depois = [
    gerar_resposta(student_model, student_tokenizer, ex["instruction"], ex["input"])
    for ex in dataset_distilacao
]

# Mostra 3 exemplos
for r, resp_depois in zip(respostas_aluno_antes[:3], respostas_aluno_depois[:3]):
    print(f"Pergunta  : {r['instruction']}")
    print(f"Esperado  : {r['output_esperado']}")
    print(f"Professor : {r['answer_professor'][:120]}")
    print(f"Aluno antes  : {r['resposta_aluno_antes'][:120]}")
    print(f"Aluno depois : {resp_depois[:120]}")
    print()

# Merge: junta antes + depois em um único arquivo de comparação
comparacao_completa = []
for r, resp_depois in zip(respostas_aluno_antes, respostas_aluno_depois):
    comparacao_completa.append({
        "id":                    r["id"],
        "instruction":           r["instruction"],
        "input":                 r["input"],
        "output_esperado":       r["output_esperado"],
        "reasoning_professor":   r["reasoning_professor"],
        "answer_professor":      r["answer_professor"],
        "resposta_aluno_antes":  r["resposta_aluno_antes"],
        "resposta_aluno_depois": resp_depois,
    })

with open(result_path("comparacao_completa_q4.json"), "w", encoding="utf-8") as f:
    json.dump(comparacao_completa, f, ensure_ascii=False, indent=2)
print(f"Comparação completa salva em {result_path('comparacao_completa_q4.json')} ({len(comparacao_completa)} exemplos)")
print("Campos por exemplo:")
print("  - instruction          : pergunta")
print("  - output_esperado      : resposta correta do benchmark")
print("  - reasoning_professor  : raciocínio gerado pelo professor")
print("  - answer_professor     : resposta do professor")
print("  - resposta_aluno_antes : aluno SEM fine-tuning")
print("  - resposta_aluno_depois: aluno APÓS destilação")

---
# PARTE 5 — Avaliação do Professor e Comparação Final
Recarrega o professor para calcular as mesmas métricas, criando o teto de referência.

## 5.1 — Métricas do Professor (referência de teto)

In [ ]:
# Libera VRAM antes de recarregar o professor
torch.cuda.empty_cache()

print(f"Recarregando professor ({TEACHER_NAME}) para avaliação final...")

bnb_config_eval = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

teacher_model_eval = AutoModelForCausalLM.from_pretrained(
    TEACHER_NAME,
    device_map="auto",
    quantization_config=bnb_config_eval,
)
teacher_model_eval.eval()

metricas_professor = calcular_metricas(teacher_model_eval, student_tokenizer, dataset_distilacao)

print("\n=== PROFESSOR (referência de teto) ===")
print(f"  Entropia Cruzada : {metricas_professor['entropia_cruzada']}")
print(f"  Perplexidade     : {metricas_professor['perplexidade']}")
print(f"  Acurácia Tokens  : {metricas_professor['acuracia_tokens']}%")

with open(result_path("metricas_professor.json"), "w", encoding="utf-8") as f:
    json.dump(metricas_professor, f, ensure_ascii=False, indent=2)
print(f"\nMétricas do professor salvas em {result_path('metricas_professor.json')}")

# Descarrega professor — SEM .to("cpu") com device_map="auto"
del teacher_model_eval
torch.cuda.empty_cache()
print("Professor descarregado.")

## 5.2 — Tabela Comparativa Final

In [ ]:
d_ec  = metricas_aluno_depois["entropia_cruzada"] - metricas_aluno_antes["entropia_cruzada"]
d_ppl = metricas_aluno_depois["perplexidade"]     - metricas_aluno_antes["perplexidade"]
d_acc = metricas_aluno_depois["acuracia_tokens"]  - metricas_aluno_antes["acuracia_tokens"]
gap   = metricas_aluno_depois["perplexidade"]     - metricas_professor["perplexidade"]

print("=" * 78)
print("         COMPARAÇÃO FINAL — DESTILAÇÃO DE CONHECIMENTO (CoT)")
print("=" * 78)
print(f"{'Métrica':<30} {'Professor':>12} {'Aluno antes':>13} {'Aluno depois':>13}")
print("-" * 78)
for m, label in [
    ("entropia_cruzada",  "Entropia Cruzada"),
    ("perplexidade",      "Perplexidade (PPL)"),
    ("acuracia_tokens",   "Acurácia Tokens (%)"),
]:
    print(f"{label:<30} {metricas_professor[m]:>12} {metricas_aluno_antes[m]:>13} {metricas_aluno_depois[m]:>13}")
print("=" * 78)

print(f"\nVariação do aluno (antes → depois):")
print(f"  Δ Perplexidade     : {d_ppl:+.2f}")
print(f"  Δ Entropia Cruzada : {d_ec:+.4f}")
print(f"  Δ Acurácia Tokens  : {d_acc:+.2f}%")
print(f"\nGap PPL aluno-depois vs professor: {gap:+.2f}")

print("\n--- Análise de Transferência de Conhecimento ---")
if d_ppl < 0:
    pct = abs(d_ppl / metricas_aluno_antes["perplexidade"]) * 100
    print(f"✓ Perplexidade REDUZIU {pct:.1f}% — houve transferência de conhecimento.")
    transferencia = f"SIM — Perplexidade reduziu {pct:.1f}%"
else:
    print("✗ Perplexidade não reduziu — considere mais épocas ou ajustar LR.")
    transferencia = "NÃO — Perplexidade não reduziu"

if d_acc > 0:
    print(f"✓ Acurácia de tokens AUMENTOU {d_acc:.2f}% — aluno aprendeu o padrão CoT do professor.")

if abs(gap) < 15:
    print("✓ Gap PPL pequeno — aluno se aproximou bem do professor.")
else:
    print(f"  Gap de {gap:.2f} PPL entre aluno e professor — diferença de capacidade dos modelos.")

# ── Salva comparação final em JSON ──────────────────────────────────────────
comparacao_final = {
    "modelos": {
        "professor": TEACHER_NAME,
        "aluno":     STUDENT_NAME,
    },
    "metricas": {
        "professor":    metricas_professor,
        "aluno_antes":  metricas_aluno_antes,
        "aluno_depois": metricas_aluno_depois,
    },
    "variacao_aluno": {
        "delta_entropia_cruzada": round(d_ec, 4),
        "delta_perplexidade":     round(d_ppl, 2),
        "delta_acuracia_tokens":  round(d_acc, 2),
    },
    "gap_vs_professor": {
        "perplexidade": round(gap, 2),
    },
    "transferencia_de_conhecimento": transferencia,
}

with open(result_path("comparacao_final_q4.json"), "w", encoding="utf-8") as f:
    json.dump(comparacao_final, f, ensure_ascii=False, indent=2)
print(f"\nComparação final salva em {result_path('comparacao_final_q4.json')}")

## 5.3 — Benchmark Qualitativo Completo (100 perguntas)

In [ ]:
print(f"Total de exemplos em comparacao_completa_q4.json: {len(comparacao_completa)}")

# Mostra 3 exemplos lado a lado das 3 colunas
print("--- 3 exemplos: professor vs aluno antes vs aluno depois ---")
for r in comparacao_completa[:3]:
    print(f"[{r['id']}] {r['instruction']}")
    print(f"  Esperado     : {r['output_esperado']}")
    print(f"  Professor    : {r['answer_professor'][:100]}")
    print(f"  Aluno antes  : {r['resposta_aluno_antes'][:100]}")
    print(f"  Aluno depois : {r['resposta_aluno_depois'][:100]}")

resultados_benchmark = [{
    "id":               r["id"],
    "instruction":      r["instruction"],
    "input":            r["input"],
    "output_esperado":  r["output_esperado"],
    "answer_professor": r["answer_professor"],
    "resposta_aluno":   r["resposta_aluno_depois"],
} for r in comparacao_completa]

with open(result_path("benchmark_q4_resultados.json"), "w", encoding="utf-8") as f:
    json.dump(resultados_benchmark, f, ensure_ascii=False, indent=2)
print(f"{result_path('benchmark_q4_resultados.json')} atualizado.")

## 5.4 — Análise de Imitação (Imitation Accuracy)

In [ ]:
def palavras_chave(texto, n=8):
    palavras = [w.lower().strip(".,;:?!\"'()[]") for w in texto.split() if len(w) > 3]
    return set(palavras[:n * 3])


concordancias = sum(
    1 for r in resultados_benchmark
    if palavras_chave(r["answer_professor"]) & palavras_chave(r["resposta_aluno"])
)
imitation_acc = concordancias / len(resultados_benchmark)

print("=" * 60)
print("   MÉTRICAS DE IMITAÇÃO — Benchmark 100 perguntas")
print("=" * 60)
print(f"Concordância de conteúdo (keyword overlap): {imitation_acc:.2%}")
print(f"  ({concordancias}/{len(resultados_benchmark)} respostas com palavras-chave em comum)")

if imitation_acc >= 0.5:
    print("✓ Aluno imitou bem o comportamento do professor (>= 50%)")
else:
    print("  Aluno ainda diverge do professor — gap de capacidade 3B → 1.5B é esperado.")

## 5.5 — Salva Resultados Finais

In [ ]:
resultados_finais = {
    "professor":  TEACHER_NAME,
    "aluno":      STUDENT_NAME,
    "dataset":    "benchmark_diarios.json",
    "n_exemplos": len(dataset_distilacao),
    "hiperparametros": {
        "epocas":         3,
        "batch_efetivo":  2 * 8,
        "learning_rate":  2e-4,
        "lora_r":         16,
        "lora_alpha":     32,
        "target_modules": ["q_proj", "v_proj"],
        "max_seq_length": 512,
    },
    "metricas_professor":    metricas_professor,
    "metricas_aluno_antes":  metricas_aluno_antes,
    "metricas_aluno_depois": metricas_aluno_depois,
    "delta_aluno": {
        "entropia_cruzada": round(d_ec, 4),
        "perplexidade":     round(d_ppl, 2),
        "acuracia_tokens":  round(d_acc, 2),
    },
    "gap_vs_professor":  {"perplexidade": round(gap, 2)},
    "imitation_accuracy": round(imitation_acc, 4),
}

with open(result_path("resultados_q4.json"), "w", encoding="utf-8") as f:
    json.dump(resultados_finais, f, ensure_ascii=False, indent=2, default=str)
print(f"Resultados salvos em {result_path('resultados_q4.json')}")

print("\n=== RESUMO FINAL ===")
print(f"Professor (3B)      -> PPL: {metricas_professor['perplexidade']} | EC: {metricas_professor['entropia_cruzada']} | Acc: {metricas_professor['acuracia_tokens']}%")
print(f"Aluno ANTES (1.5B)  -> PPL: {metricas_aluno_antes['perplexidade']} | EC: {metricas_aluno_antes['entropia_cruzada']} | Acc: {metricas_aluno_antes['acuracia_tokens']}%")
print(f"Aluno DEPOIS (1.5B) -> PPL: {metricas_aluno_depois['perplexidade']} | EC: {metricas_aluno_depois['entropia_cruzada']} | Acc: {metricas_aluno_depois['acuracia_tokens']}%")
print(f"Imitation Accuracy  : {imitation_acc:.2%}")